# BTCUSDC Order-Flow Study: Sudden Burst Clean Plot

This notebook keeps the burst study narrow: price on top, last-`N` sign imbalance in the middle, and rolling volume on the bottom. Optional panels can be turned on for buy volume, sell volume, trades per second, buy trades per second, sell trades per second, buy share, and a normalized sell/buy ratio volume panel. There is also an option to highlight imbalance-active spans in the lower panels.


## Load Data

Use the shared loader and trade-frame alignment helper, then inspect the reference day before plotting.


In [1]:
from pathlib import Path
import json
import sys
import uuid

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import HTML, display


def find_backtester_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "stats").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the exchange-data-backtester project root")


PROJECT_ROOT = find_backtester_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from stats.features import make_trade_frame
from stats.notebook import load_orderflow_day

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

EXCHANGE = "binance"
SYMBOL = "BTCUSDC"
REFERENCE_DAY = "20260223"
REPLAY_ON_GAP = "skip-segment"

_, trades, top, day_summary = load_orderflow_day(
    day=REFERENCE_DAY,
    symbol=SYMBOL,
    exchange=EXCHANGE,
    replay_on_gap=REPLAY_ON_GAP,
)

trade_frame = make_trade_frame(trades, top)

display(day_summary.to_frame("value"))
summary = pd.Series({
    "trade_frame_rows": len(trade_frame),
    "start_utc": trade_frame["ts"].min(),
    "end_utc": trade_frame["ts"].max(),
    "buy_trades": int((trade_frame["aggr_sign"] > 0).sum()),
    "sell_trades": int((trade_frame["aggr_sign"] < 0).sum()),
})
display(summary.to_frame("value"))
display(trade_frame.head())


,value
exchange,binance
symbol,BTCUSDC
day,20260223
day_dir,/Users/hoangdeveloper/PycharmProjects/exchange...
trades_rows,868008
top_rows,722893
trade_start_utc,2026-02-23 01:00:04.958000+00:00
trade_end_utc,2026-02-23 23:15:00.320000+00:00
top_start_utc,2026-02-23 01:00:04.735000+00:00
top_end_utc,2026-02-23 23:14:59.835000+00:00


,value
trade_frame_rows,868008
start_utc,2026-02-23 01:00:04.958000+00:00
end_utc,2026-02-23 23:15:00.320000+00:00
buy_trades,445726
sell_trades,422282


,ts,price,qty,aggr_sign,mid_at_book
0,2026-02-23 01:00:04.958000+00:00,66704.44,0.00009,1.0,66704.435
1,2026-02-23 01:00:04.958000+00:00,66704.44,0.00017,1.0,66704.435
2,2026-02-23 01:00:05.037000+00:00,66704.44,0.00989,1.0,66704.435
3,2026-02-23 01:00:05.087000+00:00,66704.43,0.00123,-1.0,66704.435
4,2026-02-23 01:00:05.087000+00:00,66702.01,0.00008,-1.0,66704.435


## Interactive Price

The notebook opens with midprice only. Use the checkboxes to turn on the imbalance and lower panels you want to inspect.

When enabled, the middle panel shows the rolling sign imbalance over the last `N` trades:

`signed_share_N = sum(last N trade signs) / N`

The bottom volume panel always shows total volume over the same last `N` trades. The trades-per-second panel shows how compressed the same window is in clock time. The optional sell/buy ratio volume panel is normalized to `[-1, 1]`, where `+1` is all sell volume and `-1` is all buy volume.

Use `N trades` to change the window length and `imb th` to change the cutoff for highlighting buy and sell markers. Green markers on price highlight buy-side windows; red markers highlight sell-side windows. The dashed cursor line is shared across all visible panels.


In [2]:
DEFAULT_EVENT_N = 100
DEFAULT_IMBALANCE_THRESHOLD = 0.70
MAX_PLOT_ROWS = 50_000


def _sample_evenly(frame: pd.DataFrame, *, max_rows: int) -> pd.DataFrame:
    if len(frame) <= max_rows:
        return frame.copy()
    step = int(np.ceil(len(frame) / max_rows))
    return frame.iloc[::step].copy()


def _masked_series(values: pd.Series, mask: pd.Series) -> pd.Series:
    masked = values.copy().astype(float)
    masked = masked.where(mask, np.nan)
    return masked


def build_event_sign_features(trade_frame: pd.DataFrame, *, event_n: int) -> pd.DataFrame:
    event_n = int(event_n)
    if event_n <= 1:
        raise ValueError("event_n must be greater than 1")
    if len(trade_frame) < event_n:
        raise ValueError(f"not enough trades for event_n={event_n}")

    out = trade_frame[["ts", "mid_at_book", "qty", "aggr_sign"]].copy().sort_values("ts").reset_index(drop=True)
    signs = out["aggr_sign"].to_numpy(dtype=float)
    qty = out["qty"].to_numpy(dtype=float)
    buy_qty = np.where(signs > 0, qty, 0.0)
    sell_qty = np.where(signs < 0, qty, 0.0)

    out["net_count_N"] = pd.Series(signs).rolling(event_n, min_periods=event_n).sum().to_numpy()
    out["signed_share_N"] = out["net_count_N"] / float(event_n)
    out["total_volume_N"] = pd.Series(qty).rolling(event_n, min_periods=event_n).sum().to_numpy()
    out["buy_volume_N"] = pd.Series(buy_qty).rolling(event_n, min_periods=event_n).sum().to_numpy()
    out["sell_volume_N"] = pd.Series(sell_qty).rolling(event_n, min_periods=event_n).sum().to_numpy()
    out["volume_share_N"] = (out["buy_volume_N"] - out["sell_volume_N"]) / out["total_volume_N"].replace(0.0, np.nan)
    total_signed_volume = out["sell_volume_N"] + out["buy_volume_N"]
    out["sell_buy_ratio_volume_N"] = (out["sell_volume_N"] - out["buy_volume_N"]) / total_signed_volume.replace(0.0, np.nan)
    out["buy_count_N"] = pd.Series((signs > 0).astype(float)).rolling(event_n, min_periods=event_n).sum().to_numpy()
    out["sell_count_N"] = pd.Series((signs < 0).astype(float)).rolling(event_n, min_periods=event_n).sum().to_numpy()
    event_span_sec = (out["ts"] - out["ts"].shift(event_n - 1)).dt.total_seconds()
    span_sec = event_span_sec.replace(0.0, np.nan)
    out["trades_per_sec_N"] = float(event_n) / span_sec
    out["buy_trades_per_sec_N"] = out["buy_count_N"] / span_sec
    out["sell_trades_per_sec_N"] = out["sell_count_N"] / span_sec
    out["buy_share_N"] = out["buy_count_N"] / float(event_n)
    return out.dropna(subset=["mid_at_book", "signed_share_N", "total_volume_N", "buy_volume_N", "sell_volume_N", "volume_share_N"]).reset_index(drop=True)


def make_event_sign_figure(
    features: pd.DataFrame,
    *,
    threshold: float,
    show_buy_volume: bool,
    show_sell_volume: bool,
    show_buy_minus_sell_volume: bool,
    show_ratio: bool,
    show_buy_tps: bool,
    show_sell_tps: bool,
    show_buy_share: bool,
    show_sign_imbalance: bool,
    show_imbalance_highlight: bool,
    title: str,
) -> go.Figure:
    plot_df = _sample_evenly(features, max_rows=MAX_PLOT_ROWS)
    price_df = features[["ts", "mid_at_book"]].copy().sort_values("ts")
    buy_df = features.loc[features["signed_share_N"] >= threshold].copy()
    sell_df = features.loc[features["signed_share_N"] <= -threshold].copy()
    buy_volume_df = features.loc[features["volume_share_N"] >= threshold].copy()
    sell_volume_df = features.loc[features["volume_share_N"] <= -threshold].copy()
    active_mask = plot_df["signed_share_N"].abs() >= threshold

    panel_specs = [("midprice", "mid")]
    if show_sign_imbalance:
        panel_specs.append(("last-N sign imbalance", "signed share"))
    panel_specs.extend([
        ("last-N total volume", "volume"),
        ("last-N trades per second", "trades / sec"),
    ])
    if show_buy_tps:
        panel_specs.append(("last-N buy trades per second", "buy trades / sec"))
    if show_sell_tps:
        panel_specs.append(("last-N sell trades per second", "sell trades / sec"))
    if show_buy_share:
        panel_specs.append(("last-N buy share", "buy share"))
    if show_buy_volume:
        panel_specs.append(("last-N buy volume", "buy volume"))
    if show_sell_volume:
        panel_specs.append(("last-N sell volume", "sell volume"))
    if show_buy_minus_sell_volume:
        panel_specs.append(("last-N buy volume - sell volume", "buy - sell vol"))
    if show_ratio:
        panel_specs.append(("last-N sell/buy ratio volume", "sell / buy vol"))

    row_count = len(panel_specs)
    row_heights = [0.36] + [0.64 / max(1, row_count - 1)] * (row_count - 1)

    fig = make_subplots(
        rows=row_count,
        cols=1,
        shared_xaxes=True,
        row_heights=row_heights,
        vertical_spacing=0.06,
        subplot_titles=tuple(title for title, _ in panel_specs),
    )

    fig.add_trace(
        go.Scattergl(
            x=price_df["ts"],
            y=price_df["mid_at_book"],
            mode="lines",
            line=dict(color="#111827", width=1),
            name="midprice",
            hoverinfo="skip",
        ),
        row=1,
        col=1,
    )

    for data, name, color, symbol in [
        (buy_df, "buy imbalance", "#16a34a", "circle"),
        (sell_df, "sell imbalance", "#dc2626", "circle"),
        (buy_volume_df, "buy volume imbalance", "#2563eb", "square"),
        (sell_volume_df, "sell volume imbalance", "#d97706", "square"),
    ]:
        customdata = None
        if len(data):
            customdata = np.stack([data["signed_share_N"], data["net_count_N"]], axis=-1)
        fig.add_trace(
            go.Scattergl(
                x=data["ts"],
                y=data["mid_at_book"],
                mode="markers",
                marker=dict(color=color, size=7, opacity=0.85, symbol=symbol),
                name=name,
                customdata=customdata,
                hovertemplate=(
                    "%{x}<br>mid=%{y:.2f}"
                    "<br>signed_share=%{customdata[0]:.3f}"
                    "<br>net_count=%{customdata[1]:.0f}<extra></extra>"
                ),
            ),
            row=1,
            col=1,
        )

    signed_row = 2 if show_sign_imbalance else None
    if show_sign_imbalance:
        fig.add_trace(
            go.Scattergl(
                x=plot_df["ts"],
                y=plot_df["signed_share_N"],
                mode="lines",
                line=dict(color="#2563eb", width=1),
                name="signed_share_N",
            ),
            row=2,
            col=1,
        )
        for y, color in [(threshold, "#16a34a"), (-threshold, "#dc2626"), (0.0, "#6b7280")]:
            fig.add_hline(y=y, line_width=1, line_dash="dash", line_color=color, row=2, col=1)

    def add_lower_line(row: int, y_series: pd.Series, *, color: str, name: str, title_text: str) -> None:
        base_opacity = 0.18 if show_imbalance_highlight else 1.0
        fig.add_trace(
            go.Scattergl(
                x=plot_df["ts"],
                y=y_series,
                mode="lines",
                line=dict(color=color, width=1),
                opacity=base_opacity,
                name=name if not show_imbalance_highlight else f"{name} (base)",
                hoverinfo="skip" if show_imbalance_highlight else "x+y",
            ),
            row=row,
            col=1,
        )
        if show_imbalance_highlight:
            highlighted = y_series.where(active_mask, np.nan)
            fig.add_trace(
                go.Scattergl(
                    x=plot_df["ts"],
                    y=highlighted,
                    mode="lines",
                    line=dict(color=color, width=2),
                    name=name,
                ),
                row=row,
                col=1,
            )
        fig.update_yaxes(title_text=title_text, row=row, col=1)

    current_row = 3 if show_sign_imbalance else 2
    add_lower_line(current_row, plot_df["total_volume_N"], color="#7c3aed", name="total_volume_N", title_text="volume")
    current_row += 1

    add_lower_line(current_row, plot_df["trades_per_sec_N"], color="#d97706", name="trades_per_sec_N", title_text="trades / sec")
    current_row += 1

    if show_buy_tps:
        add_lower_line(current_row, plot_df["buy_trades_per_sec_N"], color="#0f766e", name="buy_trades_per_sec_N", title_text="buy trades / sec")
        current_row += 1

    if show_sell_tps:
        add_lower_line(current_row, plot_df["sell_trades_per_sec_N"], color="#b45309", name="sell_trades_per_sec_N", title_text="sell trades / sec")
        current_row += 1

    if show_buy_share:
        add_lower_line(current_row, plot_df["buy_share_N"], color="#2563eb", name="buy_share_N", title_text="buy share")
        current_row += 1

    if show_buy_volume:
        add_lower_line(current_row, plot_df["buy_volume_N"], color="#059669", name="buy_volume_N", title_text="buy volume")
        current_row += 1

    if show_sell_volume:
        add_lower_line(current_row, plot_df["sell_volume_N"], color="#b91c1c", name="sell_volume_N", title_text="sell volume")
        current_row += 1

    if show_buy_minus_sell_volume:
        add_lower_line(current_row, plot_df["buy_volume_N"] - plot_df["sell_volume_N"], color="#0f766e", name="buy_minus_sell_volume_N", title_text="buy volume - sell volume")
        current_row += 1

    if show_ratio:
        add_lower_line(current_row, plot_df["sell_buy_ratio_volume_N"], color="#ca8a04", name="sell_buy_ratio_volume_N", title_text="sell / buy vol")
        current_row += 1

    hover_x = plot_df["ts"].iloc[0]
    for row in range(1, row_count + 1):
        yref = "y domain" if row == 1 else f"y{row} domain"
        fig.add_shape(
            type="line",
            xref="x",
            yref=yref,
            x0=hover_x,
            x1=hover_x,
            y0=0,
            y1=1,
            line=dict(color="#6b7280", width=1, dash="dash"),
            visible=False,
            layer="above",
        )

    fig.update_layout(
        height=700 + 300 * row_count,
        width=2200,
        hovermode="x",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        margin=dict(l=60, r=30, t=90, b=50),
    )
    fig.update_yaxes(title_text="mid", row=1, col=1)
    if show_sign_imbalance:
        fig.update_yaxes(title_text="signed share", row=2, col=1)
    fig.update_xaxes(title_text="UTC time", row=row_count, col=1)
    fig._guide_count = row_count
    fig._plot_title = title
    return fig


def render_shared_hover_figure(fig: go.Figure) -> None:
    wrapper_id = f"burst-clean-hover-{uuid.uuid4().hex}"
    guide_count = int(getattr(fig, "_guide_count", 1))
    graph_html = fig.to_html(
        full_html=False,
        include_plotlyjs=True,
        config={"responsive": True, "scrollZoom": True, "displaylogo": False},
    )
    plot_title = getattr(fig, "_plot_title", "")
    html = ('''
<div id="__WRAPPER__" style="margin-bottom: 1rem;">
  <div style="font-size: 18px; font-weight: 600; margin: 0 0 0.5rem 0;">__PLOT_TITLE__</div>
  __GRAPH__
</div>
<script>
(function() {
  const root = document.getElementById(__WRAPPER_JSON__);
  if (!root) {
    return;
  }

  function attach() {
    const graph = root.querySelector('.plotly-graph-div');
    if (!graph || typeof graph.on !== 'function') {
      setTimeout(attach, 50);
      return;
    }

    const guideCount = __GUIDE_COUNT__;
    const shapeStart = Array.isArray(graph.layout.shapes) ? graph.layout.shapes.length - guideCount : -1;
    if (shapeStart < 0) {
      return;
    }

    const shapeIndices = Array.from({ length: guideCount }, (_, idx) => shapeStart + idx);

    function setLine(xValue, visible) {
      const update = {};
      shapeIndices.forEach((shapeIndex) => {
        update[`shapes[${shapeIndex}].visible`] = visible;
        if (xValue !== undefined && xValue !== null) {
          update[`shapes[${shapeIndex}].x0`] = xValue;
          update[`shapes[${shapeIndex}].x1`] = xValue;
        }
      });
      Plotly.relayout(graph, update);
    }

    graph.on('plotly_hover', function(evt) {
      const point = evt && evt.points && evt.points[0];
      if (!point) {
        return;
      }
      setLine(point.x, true);
    });

    graph.on('plotly_unhover', function() {
      setLine(null, false);
    });
  }

  attach();
})();
</script>
'''
    .replace("__WRAPPER__", wrapper_id)
    .replace("__WRAPPER_JSON__", json.dumps(wrapper_id))
    .replace("__GRAPH__", graph_html)
    .replace("__PLOT_TITLE__", plot_title)
    .replace("__GUIDE_COUNT__", str(guide_count))
    )
    display(HTML(html))
def draw_event_sign_plot(
    event_n: int = DEFAULT_EVENT_N,
    threshold: float = DEFAULT_IMBALANCE_THRESHOLD,
    show_buy_volume: bool = False,
    show_sell_volume: bool = False,
    show_buy_minus_sell_volume: bool = False,
    show_ratio: bool = False,
    show_buy_tps: bool = False,
    show_sell_tps: bool = False,
    show_buy_share: bool = False,
    show_sign_imbalance: bool = False,
    show_imbalance_highlight: bool = False,
):
    features = build_event_sign_features(trade_frame, event_n=int(event_n))
    summary = pd.Series({
        "event_n": int(event_n),
        "imbalance_threshold": float(threshold),
        "show_buy_volume": bool(show_buy_volume),
        "show_sell_volume": bool(show_sell_volume),
        "show_buy_minus_sell_volume": bool(show_buy_minus_sell_volume),
        "show_ratio": bool(show_ratio),
        "show_buy_tps": bool(show_buy_tps),
        "show_sell_tps": bool(show_sell_tps),
        "show_buy_share": bool(show_buy_share),
        "show_sign_imbalance": bool(show_sign_imbalance),
        "show_imbalance_highlight": bool(show_imbalance_highlight),
        "rows": len(features),
        "buy_markers": int((features["signed_share_N"] >= threshold).sum()),
        "sell_markers": int((features["signed_share_N"] <= -threshold).sum()),
    })
    display(summary.to_frame("value"))
    fig = make_event_sign_figure(
        features,
        threshold=float(threshold),
        show_buy_volume=bool(show_buy_volume),
        show_sell_volume=bool(show_sell_volume),
        show_buy_minus_sell_volume=bool(show_buy_minus_sell_volume),
        show_ratio=bool(show_ratio),
        show_buy_tps=bool(show_buy_tps),
        show_sell_tps=bool(show_sell_tps),
        show_buy_share=bool(show_buy_share),
        show_sign_imbalance=bool(show_sign_imbalance),
        show_imbalance_highlight=bool(show_imbalance_highlight),
        title=f"{SYMBOL} {REFERENCE_DAY}: midprice and trade-flow diagnostics over last {int(event_n)} trades",
    )
    render_shared_hover_figure(fig)
    global latest_event_sign_features, latest_event_sign_config
    latest_event_sign_features = features
    latest_event_sign_config = summary
    return features


N_WIDGET = widgets.IntSlider(value=DEFAULT_EVENT_N, min=20, max=500, step=10, description="N trades", continuous_update=False)
THRESHOLD_WIDGET = widgets.FloatSlider(
    value=DEFAULT_IMBALANCE_THRESHOLD,
    min=0.10,
    max=1.00,
    step=0.05,
    description="imb th",
    continuous_update=False,
)
BUY_VOLUME_WIDGET = widgets.Checkbox(value=False, description="buy volume")
SELL_VOLUME_WIDGET = widgets.Checkbox(value=False, description="sell volume")
BUY_MINUS_SELL_VOLUME_WIDGET = widgets.Checkbox(value=False, description="buy-sell volume")
RATIO_WIDGET = widgets.Checkbox(value=False, description="sell/buy ratio volume")
BUY_TPS_WIDGET = widgets.Checkbox(value=False, description="buy tps")
SELL_TPS_WIDGET = widgets.Checkbox(value=False, description="sell tps")
BUY_SHARE_WIDGET = widgets.Checkbox(value=False, description="buy share")
SIGN_IMBALANCE_WIDGET = widgets.Checkbox(value=False, description="sign imbalance")
HIGHLIGHT_WIDGET = widgets.Checkbox(value=False, description="highlight imbalance")
REFRESH_BUTTON = widgets.Button(description="Refresh plot", button_style="primary")
REFRESH_OUTPUT = widgets.Output()


def _refresh_event_sign_plot(_=None):
    with REFRESH_OUTPUT:
        REFRESH_OUTPUT.clear_output(wait=True)
        draw_event_sign_plot(
            event_n=N_WIDGET.value,
            threshold=THRESHOLD_WIDGET.value,
            show_buy_volume=BUY_VOLUME_WIDGET.value,
            show_sell_volume=SELL_VOLUME_WIDGET.value,
            show_buy_minus_sell_volume=BUY_MINUS_SELL_VOLUME_WIDGET.value,
            show_ratio=RATIO_WIDGET.value,
            show_buy_tps=BUY_TPS_WIDGET.value,
            show_sell_tps=SELL_TPS_WIDGET.value,
            show_buy_share=BUY_SHARE_WIDGET.value,
            show_sign_imbalance=SIGN_IMBALANCE_WIDGET.value,
            show_imbalance_highlight=HIGHLIGHT_WIDGET.value,
        )


def _observe_widget_change(change):
    if change.get("name") == "value":
        _refresh_event_sign_plot()


REFRESH_BUTTON.on_click(_refresh_event_sign_plot)
for widget in [
    N_WIDGET,
    THRESHOLD_WIDGET,
    BUY_VOLUME_WIDGET,
    SELL_VOLUME_WIDGET,
    BUY_MINUS_SELL_VOLUME_WIDGET,
    RATIO_WIDGET,
    BUY_TPS_WIDGET,
    SELL_TPS_WIDGET,
    BUY_SHARE_WIDGET,
    SIGN_IMBALANCE_WIDGET,
    HIGHLIGHT_WIDGET,
]:
    widget.observe(_observe_widget_change, names="value")
display(widgets.VBox([
    widgets.HBox([N_WIDGET, THRESHOLD_WIDGET]),
    widgets.HBox([BUY_VOLUME_WIDGET, SELL_VOLUME_WIDGET, BUY_MINUS_SELL_VOLUME_WIDGET, RATIO_WIDGET]),
    widgets.HBox([BUY_TPS_WIDGET, SELL_TPS_WIDGET, BUY_SHARE_WIDGET, SIGN_IMBALANCE_WIDGET, HIGHLIGHT_WIDGET]),
    REFRESH_BUTTON,
    REFRESH_OUTPUT,
]))
_refresh_event_sign_plot()

## Microprice and Bursty Negative Volume Bins

This section focuses on the interval where count imbalance stayed positive but price still drifted lower.
The first figure isolates microprice and top-of-book pressure.
The second figure highlights 5-minute bins with negative volume imbalance and unusually high trade intensity.


In [ ]:
STUDY_START = pd.Timestamp("2026-02-23 18:00:00", tz="UTC")
STUDY_END = pd.Timestamp("2026-02-23 20:00:00", tz="UTC")
BURST_BIN = "5min"
BURST_TPS_Q = 0.90
BOOK_IMBALANCE_MA = "2min"


def build_pressure_window(trade_frame: pd.DataFrame, top_df: pd.DataFrame, *, start_ts: pd.Timestamp, end_ts: pd.Timestamp, bin_freq: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, float]:
    trade_slice = trade_frame.loc[
        (trade_frame["ts"] >= start_ts) & (trade_frame["ts"] <= end_ts),
        ["ts", "mid_at_book", "qty", "aggr_sign"],
    ].copy()
    book_slice = top_df.loc[
        (top_df["ts"] >= start_ts) & (top_df["ts"] <= end_ts),
        ["ts", "mid", "microprice", "bid1_qty", "ask1_qty", "spread_bps"],
    ].copy()
    if trade_slice.empty or book_slice.empty:
        raise ValueError("No rows found in the selected study window")

    book_denom = (book_slice["bid1_qty"] + book_slice["ask1_qty"]).replace(0.0, np.nan)
    book_slice["book_imbalance"] = (book_slice["bid1_qty"] - book_slice["ask1_qty"]) / book_denom
    book_slice["micro_mid_diff"] = book_slice["microprice"] - book_slice["mid"]
    book_slice["book_imbalance_ma"] = (
        book_slice.set_index("ts")["book_imbalance"].rolling(BOOK_IMBALANCE_MA, min_periods=25).mean().to_numpy()
    )

    trade_slice["bin"] = trade_slice["ts"].dt.floor(bin_freq)
    trade_slice["buy_qty"] = np.where(trade_slice["aggr_sign"] > 0, trade_slice["qty"], 0.0)
    trade_slice["sell_qty"] = np.where(trade_slice["aggr_sign"] < 0, trade_slice["qty"], 0.0)
    trade_slice["buy_count"] = (trade_slice["aggr_sign"] > 0).astype(float)
    trade_slice["sell_count"] = (trade_slice["aggr_sign"] < 0).astype(float)

    trade_agg = trade_slice.groupby("bin").agg(
        trades=("qty", "size"),
        buy_trades=("buy_count", "sum"),
        sell_trades=("sell_count", "sum"),
        total_qty=("qty", "sum"),
        buy_qty=("buy_qty", "sum"),
        sell_qty=("sell_qty", "sum"),
        first_mid=("mid_at_book", "first"),
        last_mid=("mid_at_book", "last"),
    )
    trade_agg["count_imbalance"] = (trade_agg["buy_trades"] - trade_agg["sell_trades"]) / trade_agg["trades"]
    trade_agg["volume_imbalance"] = (trade_agg["buy_qty"] - trade_agg["sell_qty"]) / trade_agg["total_qty"].replace(0.0, np.nan)
    trade_agg["trades_per_sec"] = trade_agg["trades"] / pd.to_timedelta(bin_freq).total_seconds()
    burst_threshold = float(trade_agg["trades_per_sec"].quantile(BURST_TPS_Q))
    trade_agg["negative_burst"] = (trade_agg["volume_imbalance"] < 0) & (trade_agg["trades_per_sec"] >= burst_threshold)
    trade_agg["ret"] = np.log(trade_agg["last_mid"] / trade_agg["first_mid"])
    return trade_slice, book_slice, trade_agg, burst_threshold


study_trade_slice, study_book_slice, study_bins, burst_threshold = build_pressure_window(
    trade_frame,
    top,
    start_ts=STUDY_START,
    end_ts=STUDY_END,
    bin_freq=BURST_BIN,
)

study_summary = pd.Series({
    "start_utc": STUDY_START,
    "end_utc": STUDY_END,
    "trade_rows": len(study_trade_slice),
    "book_rows": len(study_book_slice),
    "5m_bins": len(study_bins),
    "negative_burst_bins": int(study_bins["negative_burst"].sum()),
    "burst_threshold_tps": burst_threshold,
    "volume_imbalance_mean": float(study_bins["volume_imbalance"].mean()),
    "volume_imbalance_median": float(study_bins["volume_imbalance"].median()),
    "book_imbalance_mean": float(study_book_slice["book_imbalance"].mean()),
    "book_imbalance_ma_mean": float(study_book_slice["book_imbalance_ma"].mean()),
    "micro_mid_diff_mean": float(study_book_slice["micro_mid_diff"].mean()),
})
display(study_summary.to_frame("value"))

negative_bins = study_bins.loc[study_bins["negative_burst"], ["volume_imbalance", "trades_per_sec", "buy_qty", "sell_qty", "ret"]].copy()
display(negative_bins.sort_values(["volume_imbalance", "trades_per_sec"]).head(20))

fig_pressure = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10,
    subplot_titles=("midprice and microprice", "top-of-book imbalance, smoothed"),
)
fig_pressure.add_trace(
    go.Scattergl(
        x=study_book_slice["ts"],
        y=study_book_slice["mid"],
        mode="lines",
        line=dict(color="#111827", width=1),
        name="midprice",
    ),
    row=1,
    col=1,
)
fig_pressure.add_trace(
    go.Scattergl(
        x=study_book_slice["ts"],
        y=study_book_slice["microprice"],
        mode="lines",
        line=dict(color="#2563eb", width=1),
        name="microprice",
    ),
    row=1,
    col=1,
)
fig_pressure.add_trace(
    go.Scattergl(
        x=study_book_slice["ts"],
        y=study_book_slice["micro_mid_diff"],
        mode="lines",
        line=dict(color="#7c3aed", width=1),
        name="microprice - mid",
    ),
    row=2,
    col=1,
)
fig_pressure.add_trace(
    go.Scattergl(
        x=study_book_slice["ts"],
        y=study_book_slice["book_imbalance"],
        mode="lines",
        line=dict(color="#dc2626", width=1),
        opacity=0.25,
        name="top-of-book imbalance (raw)",
    ),
    row=2,
    col=1,
)
fig_pressure.add_trace(
    go.Scattergl(
        x=study_book_slice["ts"],
        y=study_book_slice["book_imbalance_ma"],
        mode="lines",
        line=dict(color="#dc2626", width=2),
        name=f"top-of-book imbalance MA ({BOOK_IMBALANCE_MA})",
    ),
    row=2,
    col=1,
)
fig_pressure.add_hline(y=0.0, line_width=1, line_dash="dash", line_color="#6b7280", row=2, col=1)
fig_pressure.update_layout(
    height=980,
    width=1200,
    hovermode="x unified",
    title=f"{SYMBOL} {REFERENCE_DAY}: microprice and top-of-book pressure, {STUDY_START} to {STUDY_END}",
    margin=dict(l=60, r=30, t=90, b=50),
)
fig_pressure.update_yaxes(title_text="price", row=1, col=1)
fig_pressure.update_yaxes(title_text="pressure", row=2, col=1)
fig_pressure.update_xaxes(title_text="UTC time", row=2, col=1)
display(fig_pressure)

burst_colors = np.where(
    study_bins["negative_burst"],
    "#b91c1c",
    np.where(study_bins["volume_imbalance"] < 0, "#fecaca", "#60a5fa"),
)
fig_burst = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.68, 0.32],
    vertical_spacing=0.10,
    subplot_titles=("5-minute volume imbalance", "5-minute trades per second"),
)
fig_burst.add_trace(
    go.Bar(
        x=study_bins.index,
        y=study_bins["volume_imbalance"],
        marker=dict(color=burst_colors),
        name="volume imbalance",
    ),
    row=1,
    col=1,
)
fig_burst.add_hline(y=0.0, line_width=1, line_dash="dash", line_color="#6b7280", row=1, col=1)
if study_bins["negative_burst"].any():
    fig_burst.add_trace(
        go.Scattergl(
            x=study_bins.index[study_bins["negative_burst"]],
            y=study_bins.loc[study_bins["negative_burst"], "volume_imbalance"],
            mode="markers",
            marker=dict(color="#7f1d1d", size=10, symbol="x"),
            name="negative burst bins",
        ),
        row=1,
        col=1,
    )
fig_burst.add_trace(
    go.Scattergl(
        x=study_bins.index,
        y=study_bins["trades_per_sec"],
        mode="lines+markers",
        line=dict(color="#d97706", width=1),
        marker=dict(size=5),
        name="trades per second",
    ),
    row=2,
    col=1,
)
fig_burst.add_hline(y=burst_threshold, line_width=1, line_dash="dash", line_color="#6b7280", row=2, col=1)
if study_bins["negative_burst"].any():
    fig_burst.add_trace(
        go.Scattergl(
            x=study_bins.index[study_bins["negative_burst"]],
            y=study_bins.loc[study_bins["negative_burst"], "trades_per_sec"],
            mode="markers",
            marker=dict(color="#b91c1c", size=10, symbol="x"),
            name="negative burst bins",
        ),
        row=2,
        col=1,
    )
fig_burst.update_layout(
    height=980,
    width=1200,
    hovermode="x unified",
    title=f"{SYMBOL} {REFERENCE_DAY}: bursty negative volume bins, {STUDY_START} to {STUDY_END}",
    margin=dict(l=60, r=30, t=90, b=50),
)
fig_burst.update_yaxes(title_text="volume imbalance", row=1, col=1)
fig_burst.update_yaxes(title_text="trades / sec", row=2, col=1)
fig_burst.update_xaxes(title_text="UTC time", row=2, col=1)
display(fig_burst)


,value
start_utc,2026-02-23 18:00:00+00:00
end_utc,2026-02-23 20:00:00+00:00
trade_rows,95447
book_rows,67508
5m_bins,24
negative_burst_bins,1
burst_threshold_tps,19.757333
volume_imbalance_mean,0.038565
volume_imbalance_median,0.090359
book_imbalance_mean,0.005081


,volume_imbalance,trades_per_sec,buy_qty,sell_qty,ret
bin,,,,,
2026-02-23 19:35:00+00:00,-0.320705,20.366667,17.01011,33.07155,-0.003114
